In [1]:
from pathlib import Path
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.llms import Ollama

In [2]:
VECTORSTORE_DIR = Path("../vectorstore/chroma_db")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = Chroma(
    persist_directory=str(VECTORSTORE_DIR),
    embedding_function=embeddings
)

print(f"Vectorstore loaded: {vectorstore._collection.count()} chunks ready.")

/Users/prasunamannava/clinical-prior-auth-assistant/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/prasunamannava/clinical-prior-auth-assistant/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vectorstore loaded: 2511 chunks ready.


In [3]:
llm = Ollama(
    model="llama3",
    temperature=0,
)

print("LLM connected.")

LLM connected.


In [4]:
prompt_template = """You are a clinical prior authorization specialist with expertise in 
Medi-Cal, Blue Shield of California, and Aetna insurance policies.

Use ONLY the context provided below to answer the question. If the answer is not 
in the context, say "I don't have enough information in the loaded documents to answer this question."

Do not make up information. Always cite which payer or document your answer is based on.

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

print("Prompt template defined.")

Prompt template defined.


In [5]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

print("RAG chain ready.")

RAG chain ready.


In [6]:
query = "Does Medi-Cal require a TAR for chromosomal microarray testing in a child with autism?"

result = qa_chain.invoke({"query": query})

print(f"Question: {query}")
print(f"\nAnswer:\n{result['result']}")
print(f"\nSources:")
for doc in result['source_documents']:
    print(f"  - {Path(doc.metadata['source']).name}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Question: Does Medi-Cal require a TAR for chromosomal microarray testing in a child with autism?

Answer:
I don't have enough information in the loaded documents to answer this question. The provided context only discusses Blue Shield of California's policy on chromosomal microarray analysis, and does not mention Medi-Cal or its requirements for prior authorization. Therefore, I cannot provide an answer based on the given context.

Sources:
  - blue_shield_ca_genetic_testing_dev_delay.pdf
  - blue_shield_ca_genetic_testing_dev_delay.pdf
  - blue_shield_ca_genetic_testing_dev_delay.pdf
  - blue_shield_ca_genetic_testing_dev_delay.pdf
  - blue_shield_ca_genetic_testing_dev_delay.pdf


In [7]:
def get_retriever(payer: str = None):
    """
    Returns a retriever filtered by payer source document.
    payer options: 'medi_cal', 'blue_shield', 'aetna', None (all sources)
    """
    payer_map = {
        "medi_cal": ["medi_cal_molecular_pathology.pdf", 
                     "medi_cal_genetic_counseling.pdf",
                     "medi_cal_mckt_provider_training.pdf"],
        "blue_shield": ["blue_shield_ca_genetic_testing_dev_delay.pdf"],
        "aetna": ["aetna_genetic_testing_cpb0140.pdf"]
    }

    if payer and payer in payer_map:
        search_kwargs = {
            "k": 5,
            "filter": {"source": {"$in": [
                f"../data/{f}" for f in payer_map[payer]
            ]}}
        }
    else:
        search_kwargs = {"k": 5}

    return vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs=search_kwargs
    )

print("Payer-filtered retriever defined.")

Payer-filtered retriever defined.


In [8]:
qa_chain_medical = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=get_retriever("medi_cal"),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

query = "Does Medi-Cal require a TAR for chromosomal microarray testing in a child with autism?"

result = qa_chain_medical.invoke({"query": query})

print(f"Question: {query}")
print(f"\nAnswer:\n{result['result']}")
print(f"\nSources:")
for doc in result['source_documents']:
    print(f"  - {Path(doc.metadata['source']).name}")

Question: Does Medi-Cal require a TAR for chromosomal microarray testing in a child with autism?

Answer:
I don't have enough information in the loaded documents to answer this question.

The provided context only mentions TAR requirements for prenatal testing of fetus and genomic hybridization (CGH) microarray analysis, but not specifically for chromosomal microarray testing in children. Therefore, I cannot determine whether Medi-Cal requires a TAR for this specific scenario.

Sources:
  - medi_cal_molecular_pathology.pdf
  - medi_cal_molecular_pathology.pdf
  - medi_cal_mckt_provider_training.pdf
  - medi_cal_mckt_provider_training.pdf
  - medi_cal_molecular_pathology.pdf


In [9]:
# Check what chunks are actually being retrieved
query = "Does Medi-Cal require a TAR for chromosomal microarray testing in a child with autism?"

retriever = get_retriever("medi_cal")
docs = retriever.invoke(query)

print(f"Retrieved {len(docs)} chunks:\n")
for i, doc in enumerate(docs, 1):
    print(f"--- Chunk {i} ---")
    print(f"Source: {Path(doc.metadata['source']).name}")
    print(f"Content:\n{doc.page_content}")
    print()

Retrieved 5 chunks:

--- Chunk 1 ---
Source: medi_cal_molecular_pathology.pdf
Content:
Biomarker and Pharmacogenetic Testing 
Medi -Cal covers medically necessary biomarker and pharmacogenomic testing, as 
described in the manual section Proprietary Laboratory Analyses (PLA) . Medi -Cal may not 
cover all CPT and HCPCS codes associated with a particular biomarker or 
pharmacogenomic test. As such , the particular biomarker or pharmacogenomic test code 
may be covered with an approved TAR if medical necessity is established, as described in

--- Chunk 2 ---
Source: medi_cal_molecular_pathology.pdf
Content:
genomic hybridization 
(CGH) microarray 
analysis Yes A TAR requires documentation of all 
of the following criteria for each 
indication: 
For Prenatal Testing of Fetus: 
1. Member has received 
pre-test genetic counseling 
and will receive post -test 
genetic counseling, and 
2. One of the following criteria 
must be met (a thru c): 
a. Prenatal ultrasound 
identified one or more 
s

In [10]:
retriever_large_k = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 10,
        "filter": {"source": {"$in": [
            "../data/medi_cal_molecular_pathology.pdf",
            "../data/medi_cal_genetic_counseling.pdf",
            "../data/medi_cal_mckt_provider_training.pdf"
        ]}}
    }
)

docs = retriever_large_k.invoke(query)

print(f"Retrieved {len(docs)} chunks:\n")
for i, doc in enumerate(docs, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content)
    print()

Retrieved 10 chunks:

--- Chunk 1 ---
Biomarker and Pharmacogenetic Testing 
Medi -Cal covers medically necessary biomarker and pharmacogenomic testing, as 
described in the manual section Proprietary Laboratory Analyses (PLA) . Medi -Cal may not 
cover all CPT and HCPCS codes associated with a particular biomarker or 
pharmacogenomic test. As such , the particular biomarker or pharmacogenomic test code 
may be covered with an approved TAR if medical necessity is established, as described in

--- Chunk 2 ---
genomic hybridization 
(CGH) microarray 
analysis Yes A TAR requires documentation of all 
of the following criteria for each 
indication: 
For Prenatal Testing of Fetus: 
1. Member has received 
pre-test genetic counseling 
and will receive post -test 
genetic counseling, and 
2. One of the following criteria 
must be met (a thru c): 
a. Prenatal ultrasound 
identified one or more 
structural abnormalities in 
the fetus, or 
b. Member is undergoing 
invasive diagnostic fetal

--- 

In [11]:
# Direct keyword search in vectorstore
results = vectorstore.similarity_search(
    "chromosomal microarray autism spectrum disorder pediatric TAR",
    k=5,
    filter={"source": "../data/medi_cal_molecular_pathology.pdf"}
)

for i, doc in enumerate(results, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content)
    print()

--- Chunk 1 ---
b. Multiple congenital 
anomalies without an 
established diagnosis, or 
c. Autism spectrum disorder 
with no identifiable cause, 
or 
d. Findings suggestive of 
primary 
immunodeficiency, or 
e. Congenital heart disease Once -in-a-
lifetime. 
A TAR/SAR 
may override 
the frequency 
limit.

--- Chunk 2 ---
no identifiable cause, or 
b. Multiple congenital 
anomalies without an 
established diagnosis, or 
c. Autism spectrum disorder 
with no identifiable cause, 
or 
d. Findings suggestive of 
primary 
immunodeficiency, or 
e. Congenital heart disease Once -in-a-
lifetime. 
A TAR/SAR 
may override 
the frequency 
limit.

--- Chunk 3 ---
vi. Period of unexplained developmental regression that is unrelated to epilepsy or 
autism spectrum disorder 
vii. Laboratory findings suggestive of an inherited metabolic disorder (for example, 
acidemia, hyperammonemia, mitochondrial disorders, etc.) 
Frequency limit for 81425: once in a lifetime, do not allow TAR/SAR override. 
81426 


In [12]:
# Rewritten query using clinical terminology matching the document language
query_rewritten = "chromosomal microarray TAR required autism spectrum disorder congenital anomalies pediatric Medi-Cal CPT 81228 81229"

qa_chain_medi_cal = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=get_retriever("medi_cal"),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

result = qa_chain_medi_cal.invoke({"query": query_rewritten})

print(f"Question: Does Medi-Cal require a TAR for chromosomal microarray testing in a child with autism?")
print(f"\nAnswer:\n{result['result']}")
print(f"\nSources:")
for doc in result['source_documents']:
    print(f"  - {Path(doc.metadata['source']).name}")

Question: Does Medi-Cal require a TAR for chromosomal microarray testing in a child with autism?

Answer:
Based on the provided context, I can answer your question.

For a TAR/SAR to be required for CPT codes 81228 and 81229, the following criteria must be met:

* For Prenatal Testing of Fetus:
	1. Member has received pre-test genetic counseling and will receive post-test genetic counseling, and
	2. One of the following criteria must be met (a thru c):
		a. Prenatal ultrasound identified one or more structural abnormalities in the fetus, or
		b. Member is undergoing invasive diagnostic fetal testing.

However, since the question mentions autism spectrum disorder and congenital anomalies without an established diagnosis, I would say that a TAR/SAR may override the frequency limit for CPT codes 81228 and 81229, as per Medi-Cal policy (documented in the context).

So, to answer your question:

* For pediatric patients with autism spectrum disorder or multiple congenital anomalies without 

In [13]:
query_rewritten = "molecular pathology genetic testing prior authorization TAR required Medi-Cal CPT codes"

qa_chain_medi_cal = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=get_retriever("medi_cal"),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

result = qa_chain_medi_cal.invoke({"query": query_rewritten})

print(f"Question: Does Medi-Cal require prior authorization for molecular pathology genetic testing?")
print(f"\nAnswer:\n{result['result']}")
print(f"\nSources:")
for doc in result['source_documents']:
    print(f"  - {Path(doc.metadata['source']).name}")

Question: Does Medi-Cal require prior authorization for molecular pathology genetic testing?

Answer:
Based on the provided context, I can answer that:

* For CPT code 81351 (targeted sequence analysis), a TAR is not required.
* For CPT code 81462 (DNA and RNA analysis, copy number variants and rearrangements), a TAR is required for Medi-Cal. The TAR criteria include: 
	1. Member has a diagnosis of non-small cell lung cancer
	2. Member is medically unable to undergo invasive biopsy or tumor tissue testing is not feasible
	3. Management is contingent on the test results

Note that this information is based on the Molecular Pathology CPT Codes, TAR and Billing Information document, which appears to be specific to Medi-Cal.

Sources:
  - medi_cal_molecular_pathology.pdf
  - medi_cal_molecular_pathology.pdf
  - medi_cal_molecular_pathology.pdf
  - medi_cal_molecular_pathology.pdf
  - medi_cal_molecular_pathology.pdf


In [16]:
query_rewritten = "Blue Shield California genetic testing medical necessity criteria covered indications chromosomal microarray developmental delay autism prior authorization requirements"
qa_chain_blue_shield = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=get_retriever("blue_shield"),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

result = qa_chain_blue_shield.invoke({"query": query_rewritten})

print(f"Question: What are Blue Shield of California's coverage criteria for genetic testing in children with developmental delay or autism?")
print(f"\nAnswer:\n{result['result']}")
print(f"\nSources:")
for doc in result['source_documents']:
    print(f"  - {Path(doc.metadata['source']).name}")

Question: What are Blue Shield of California's coverage criteria for genetic testing in children with developmental delay or autism?

Answer:
According to the provided context, Blue Shield of California's policy statement for Genetic Testing for Developmental Delay/Intellectual Disability, Autism Spectrum Disorder, and Congenital Anomalies (2.04.59) states that:

"I. Chromosomal microarray analysis may be considered medically necessary for the diagnosis and management of developmental delay/intellectual disability, autism spectrum disorder, and congenital anomalies."

This information is based on Page 31 of the document.

As a clinical prior authorization specialist with expertise in Blue Shield of California policies, I can confirm that chromosomal microarray analysis is covered under this policy for the specified indications.

Sources:
  - blue_shield_ca_genetic_testing_dev_delay.pdf
  - blue_shield_ca_genetic_testing_dev_delay.pdf
  - blue_shield_ca_genetic_testing_dev_delay.pdf
  -

In [15]:
query_rewritten = "prior authorization TAR required genetic testing chromosomal microarray Medi-Cal Blue Shield California medical necessity criteria"
qa_chain_cross_payer = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 8}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

result = qa_chain_cross_payer.invoke({"query": query_rewritten})

print(f"Question: Compare Medi-Cal and Blue Shield of California prior authorization requirements for genetic testing.")
print(f"\nAnswer:\n{result['result']}")
print(f"\nSources:")
for doc in result['source_documents']:
    print(f"  - {Path(doc.metadata['source']).name}")

Question: Compare Medi-Cal and Blue Shield of California prior authorization requirements for genetic testing.

Answer:
Based on the provided context from Blue Shield of California's policy 2.04.59 Genetic Testing for Developmental Delay/Intellectual Disability, Autism Spectrum Disorder, and Congenital Anomalies, a TAR (Prior Authorization Request) is required for genetic testing using chromosomal microarray analysis.

The medical necessity criteria for Medi-Cal are not explicitly stated in the provided context. However, according to Blue Shield of California's policy, a TAR requires documentation of all the following criteria:

1. Member has received pre-test genetic counseling and will receive post-test genetic counseling
2. One of the following criteria must be met:
   a. Prenatal ultrasound identified one or more structural abnormalities in the fetus
   b. Member is undergoing invasive diagnostic fetal testing

These criteria are specific to prenatal testing, but it's possible that